In [6]:
from pyspark.sql import SparkSession
import getpass

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Colab_Spark_RDD") \
    .getOrCreate()

# Lấy SparkContext từ SparkSession để dùng RDD
sc = spark.sparkContext

print("Spark version:", sc.version)


Spark version: 4.0.1


# **Bài 2: Phân Tích Đánh Giá Theo Thể Loại**

In [7]:
movies_rdd = sc.textFile("data/movies.txt")
ratings_rdd = sc.textFile("data/ratings_*.txt")
user_rdd = sc.textFile("data/users.txt")
occupation_rdd = sc.textFile("data/occupations.txt")

In [8]:
movies_rdd.take(5)

['1001,The Godfather (1972),Crime|Drama',
 '1002,The Shawshank Redemption (1994),Drama',
 "1003,Schindler's List (1993),Biography|Drama|History",
 '1004,Raging Bull (1980),Biography|Drama|Sport',
 '1005,Casablanca (1942),Drama|Romance|War']

In [18]:
ratings_rdd.take(5)

['7,1020,4.5,1577836800',
 '23,1015,3.5,1577923200',
 '45,1030,4.0,1578009600',
 '12,1047,3.0,1578096000',
 '38,1012,4.5,1578182400']

In [ ]:
#join movie và ratings
movies_mapped = movies_rdd.map(lambda x : (x.split(",")[0], x.split(",")[2]))
ratings_mapped = ratings_rdd.map(lambda x : (x.split(",")[1], float(x.split(",")[2])))
joined_rdd = ratings_mapped.join(movies_mapped)
joined_rdd.take(20)


[('1020', (4.5, 'Family|Sci-Fi')),
 ('1020', (3.5, 'Family|Sci-Fi')),
 ('1020', (3.5, 'Family|Sci-Fi')),
 ('1020', (3.5, 'Family|Sci-Fi')),
 ('1020', (3.5, 'Family|Sci-Fi')),
 ('1020', (3.5, 'Family|Sci-Fi')),
 ('1020', (3.5, 'Family|Sci-Fi')),
 ('1020', (3.0, 'Family|Sci-Fi')),
 ('1020', (4.5, 'Family|Sci-Fi')),
 ('1020', (3.0, 'Family|Sci-Fi')),
 ('1020', (4.0, 'Family|Sci-Fi')),
 ('1020', (3.5, 'Family|Sci-Fi')),
 ('1020', (4.5, 'Family|Sci-Fi')),
 ('1020', (3.0, 'Family|Sci-Fi')),
 ('1020', (4.0, 'Family|Sci-Fi')),
 ('1020', (3.5, 'Family|Sci-Fi')),
 ('1020', (4.5, 'Family|Sci-Fi')),
 ('1020', (3.0, 'Family|Sci-Fi')),
 ('1015', (3.5, 'Drama|Film-Noir')),
 ('1015', (4.5, 'Drama|Film-Noir'))]

In [30]:
#Mapper tach genre
genre_mapped = joined_rdd.flatMap(lambda x : [(genre, (x[1][0],1)) for genre in x[1][1].split("|")])
genre_mapped.take(10)

[('Family', (4.5, 1)),
 ('Sci-Fi', (4.5, 1)),
 ('Family', (3.5, 1)),
 ('Sci-Fi', (3.5, 1)),
 ('Family', (3.5, 1)),
 ('Sci-Fi', (3.5, 1)),
 ('Family', (3.5, 1)),
 ('Sci-Fi', (3.5, 1)),
 ('Family', (3.5, 1)),
 ('Sci-Fi', (3.5, 1))]

In [34]:
#reduce tinh avg va tong so rating theo genre
genre_reduced = genre_mapped.reduceByKey(lambda a,b : (a[0]+b[0], a[1]+b[1]))
genre_reduced.take(10)
genre_avg = genre_reduced.mapValues(lambda x : (x[0]/x[1], x[1]))
genre_avg.take(10)

[('Family', (3.6666666666666665, 18)),
 ('Sci-Fi', (3.7314814814814814, 54)),
 ('Drama', (3.7578125, 128)),
 ('Film-Noir', (4.357142857142857, 7)),
 ('Crime', (3.8095238095238093, 42)),
 ('Thriller', (3.7037037037037037, 27)),
 ('Biography', (3.56, 25)),
 ('Horror', (4.0, 2)),
 ('Mystery', (4.0, 2)),
 ('Action', (3.712962962962963, 54))]

In [35]:
def format_result(record):
    data = record[1]
    return f"{record[0]} - AverageRating: {data[0]:.2f} (TotalRatings: {data[1]})"

formatted_results = genre_avg.map(format_result)
formatted_results.collect()

['Family - AverageRating: 3.67 (TotalRatings: 18)',
 'Sci-Fi - AverageRating: 3.73 (TotalRatings: 54)',
 'Drama - AverageRating: 3.76 (TotalRatings: 128)',
 'Film-Noir - AverageRating: 4.36 (TotalRatings: 7)',
 'Crime - AverageRating: 3.81 (TotalRatings: 42)',
 'Thriller - AverageRating: 3.70 (TotalRatings: 27)',
 'Biography - AverageRating: 3.56 (TotalRatings: 25)',
 'Horror - AverageRating: 4.00 (TotalRatings: 2)',
 'Mystery - AverageRating: 4.00 (TotalRatings: 2)',
 'Action - AverageRating: 3.71 (TotalRatings: 54)',
 'Adventure - AverageRating: 3.63 (TotalRatings: 83)',
 'Fantasy - AverageRating: 3.86 (TotalRatings: 29)']